# Semantic Gaussian Labeling Pipeline

This is the production semantic stage for the Gaussian-centric pipeline. It does **not** build replacement mesh geometry or lift every 2D mask pixel into a separate 3D reconstruction.

Instead, it preserves and labels the Gaussians that already exist in the coherent room reconstruction:

1. Load `splat.ply` Gaussian centers, or `sparse_pc.ply` if configured.
2. Project those 3D points into every camera frame from `transforms.json`.
3. Check which projected points land inside 2D segmentation masks.
4. Optionally reject occluded points using rendered metric depth maps.
5. Vote labels onto Gaussians, append semantic metadata without changing Gaussian coordinates or appearance, cluster labeled centers into object proxies, and export viewer-compatible files.

Depth maps are used only as a visibility/occlusion test. They are never treated as a replacement surface. The visibility test is hybrid by default: rendered depth is trusted when it agrees, and a self z-buffer from the target Gaussian centers fills holes where rendered depth misses room content. The primary output is `splat_labeled.ply`; the colored point cloud and bounding boxes are diagnostic/editing overlays.


## 1. Mount Drive

In [ ]:
# --- Cell explanation -----------------------------------------------------
# This cell connects the Colab runtime to Google Drive.
# The project is edited locally on the Mac, but the real scan data and generated outputs
# live under /content/drive/MyDrive/room-model-project when the notebook runs in Colab.
# Running this cell is required before any path under /content/drive/... can be read or written.
# ------------------------------------------------------------------------

from google.colab import drive

drive.mount('/content/drive')


## 2. Install Dependencies

In [ ]:
# --- Cell explanation -----------------------------------------------------
# This cell installs the Python packages used by the geometry-labeling pipeline.
# opencv-python reads RGB frames, masks, and depth images.
# matplotlib is available for optional inspection/plots.
# pandas writes CSV reports and summarizes masks/labels.
# tqdm shows progress bars for long per-frame/per-point loops.
# scikit-learn provides DBSCAN clustering for grouping labeled points into object proxies.
# plyfile reads and writes PLY files, including Gaussian splat PLY vertex fields.
# ------------------------------------------------------------------------

!pip install -q opencv-python matplotlib pandas tqdm scikit-learn plyfile


## 3. Configure Paths and Thresholds

In [ ]:
# --- Cell explanation -----------------------------------------------------
# This is the main configuration and setup cell.
# It imports libraries, defines the Google Drive project paths, creates output/report folders,
# configures logging, and exposes the thresholds that control projection behavior.
# The most important knobs are:
# - TARGET_POINT_SOURCE: which existing 3D geometry receives labels, usually splat.ply.
# - VISIBILITY_MODE: how projected 3D points are tested for camera visibility.
# - DEPTH_* and SELF_ZBUFFER_* tolerances: how strict occlusion filtering is.
# - *_VOTE_* multipliers: how strongly different visibility/label sources vote.
# - GAUSSIAN_FOOTPRINT_* controls: sample each projected Gaussian footprint instead of a single
#   center pixel, closer to projecting segmented images onto splats.
# - LABEL_FREQUENCY_* and MASK_AREA_* weights: normalize common/huge masks so labels like wall
#   cannot dominate only because they appear often or cover most pixels.
# - SURFACE_STRUCTURAL_LABELS and STRUCTURAL_VOTE_MULTIPLIERS: prevent wall/floor/ceiling
#   from overwhelming object labels such as bed, desk, pillow, fan, and door.
# - OUTPUTS: every downstream JSON/CSV/PLY artifact written by the pipeline.
# The fail() helper records fatal errors in Drive before raising, so failed Colab runs leave a trace.
# ------------------------------------------------------------------------

from pathlib import Path
import json
import logging
import math
import re
from collections import defaultdict
from datetime import datetime

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from plyfile import PlyData, PlyElement
from sklearn.cluster import DBSCAN
from tqdm.auto import tqdm

PROJECT_ROOT = Path('/content/drive/MyDrive/room-model-project')
SCAN_ID = 'scan_001'
SCAN_DIR = PROJECT_ROOT / 'scans' / SCAN_ID

SEM_DIR = SCAN_DIR / 'semantic_recon'
DETECTION_DIR = SEM_DIR / 'detections'
PROJECTION_DIR = SEM_DIR / 'projection'
POINTS_DIR = PROJECTION_DIR / 'point_clouds'
REPORT_DIR = SCAN_DIR / 'reports' / 'projection'
FATAL_DIR = SCAN_DIR / 'reports' / 'fatal_errors'

for directory in [PROJECTION_DIR, POINTS_DIR, REPORT_DIR, FATAL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

LOG_PATH = REPORT_DIR / 'geometry_labeling.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
    handlers=[logging.FileHandler(LOG_PATH, mode='w'), logging.StreamHandler()]
)
logger = logging.getLogger('geometry_labeling')

def fail(message):
    FATAL_DIR.mkdir(parents=True, exist_ok=True)
    fatal_path = FATAL_DIR / 'fatal_error.log'
    with fatal_path.open('a') as f:
        f.write(f'[{datetime.now().isoformat(timespec="seconds")}] geometry_labeling: {message}\n')
    raise RuntimeError(message)

# Main behavior.
PIPELINE_VERSION = '2026-07-25-gaussian-footprint-mask-baking'
TARGET_POINT_SOURCE = 'splat'  # 'splat', 'sparse', or 'auto'. Splat uses Gaussian center XYZ coordinates.
# Visibility strategy. Hybrid is the recommended mode for room-scale scans:
# 1) accept rendered depth matches when available,
# 2) fall back to a self z-buffer made from the target geometry,
# 3) optionally allow low-weight no-depth votes only when explicitly enabled.
VISIBILITY_MODE = 'hybrid'  # 'hybrid', 'rendered_depth', 'self_zbuffer', or 'none'
ALLOW_NO_DEPTH_FRAME_VOTES = False

# Runtime controls. Lower MAX_POINTS_TO_LABEL or MAX_FRAMES_TO_PROCESS for a quick dry run.
MAX_POINTS_TO_LABEL = None
MAX_FRAMES_TO_PROCESS = None
MAX_MASKS_PER_FRAME = None
POINT_CHUNK_SIZE = 200_000
RANDOM_SEED = 42

# Nerfstudio transforms use OpenGL-style camera coordinates by default: -Z is forward and +Y is up.
# Use 'opencv' only if transforms.json was written with +Z-forward OpenCV camera coordinates.
CAMERA_CONVENTION = 'nerfstudio_opengl'  # 'nerfstudio_opengl' or 'opencv'

# Gaussian footprint semantic baking. Instead of using only the projected center pixel, the voting
# cell samples several pixels across the Gaussian's approximate screen-space footprint. This is an
# approximation of projecting segmented images onto splats by rendered footprint contribution.
ENABLE_GAUSSIAN_FOOTPRINT_VOTING = True
GAUSSIAN_SCALE_IS_LOG = True
GAUSSIAN_FOOTPRINT_STDDEV_MULTIPLIER = 2.0
GAUSSIAN_FOOTPRINT_FALLBACK_WORLD_RADIUS_M = 0.015
GAUSSIAN_FOOTPRINT_MIN_RADIUS_PX = 1.0
GAUSSIAN_FOOTPRINT_MAX_RADIUS_PX = 18.0
GAUSSIAN_FOOTPRINT_SAMPLE_PATTERN = 'center_8'
GAUSSIAN_FOOTPRINT_MIN_MASK_FRACTION = 0.20
GAUSSIAN_FOOTPRINT_REPORT_LABELS = {'bed', 'wall', 'floor', 'ceiling', 'desk', 'pillow', 'fan', 'door'}

# Visibility thresholds. Depth is in camera-space meters from Nerfstudio depth render outputs.
MIN_DEPTH_M = 0.05
MAX_DEPTH_M = 20.0
DEPTH_ABS_TOLERANCE_M = 0.18
DEPTH_REL_TOLERANCE = 0.08
DEPTH_PATCH_RADIUS_PX = 1

# Self z-buffer visibility catches valid projected points when rendered depth is missing/noisy.
SELF_ZBUFFER_ABS_TOLERANCE_M = 0.25
SELF_ZBUFFER_REL_TOLERANCE = 0.10
SELF_ZBUFFER_RADIUS_PX = 2
RENDERED_DEPTH_VOTE_WEIGHT = 1.0
SELF_ZBUFFER_VOTE_WEIGHT = 0.65
NO_DEPTH_VOTE_WEIGHT = 0.35

# Mask dilation is role-aware. Dilation helps tight small-object masks, but structural masks
# should not expand because wall/floor/ceiling masks are already broad.
SMALL_MASK_DILATION_PIXELS = 2
STRUCTURAL_MASK_DILATION_PIXELS = 0
WALL_MASK_DILATION_PIXELS = 0
SMALL_OBJECT_SUPPRESSION_DILATION_PIXELS = 4

# Role/label competition weights. Structural masks are intentionally conservative so broad wall
# masks cannot drown out bed, desk, pillow, fan, door, etc.
SMALL_OBJECT_VOTE_MULTIPLIER = 1.50
SURFACE_STRUCTURAL_LABELS = {'wall', 'floor', 'ceiling'}
STRUCTURAL_VOTE_MULTIPLIERS = {
    'wall': 0.05,
    'floor': 0.85,
    'ceiling': 0.85,
}
DEFAULT_STRUCTURAL_VOTE_MULTIPLIER = 0.25

# Label-frequency normalization gives each semantic label a comparable total voting budget.
# The default power 0.5 means 1/sqrt(N_label), normalized around the median label count.
# Set LABEL_FREQUENCY_POWER = 1.0 for exact inverse-frequency behavior.
ENABLE_LABEL_FREQUENCY_NORMALIZATION = True
LABEL_FREQUENCY_POWER = 0.5
LABEL_FREQUENCY_MIN_WEIGHT = 0.20
LABEL_FREQUENCY_MAX_WEIGHT = 3.00

# Mask-area normalization downweights huge masks such as wall and upweights compact masks.
# The reference fraction is the mask area that receives weight 1 before clamping.
ENABLE_MASK_AREA_NORMALIZATION = True
MASK_AREA_REFERENCE_FRACTION = 0.04
MASK_AREA_POWER = 0.5
MASK_AREA_MIN_WEIGHT = 0.20
MASK_AREA_MAX_WEIGHT = 3.00

# Object masks like bed/desk/pillow should correspond to foreground geometry, not the wall behind
# the object. This gate rejects object masks whose reconstructed depth is not closer than a ring
# around the mask. If the object is missing from splat.ply, this prevents labeling background points
# as that object.
ENABLE_OBJECT_FOREGROUND_GEOMETRY_CHECK = True
OBJECT_FOREGROUND_MIN_DELTA_M = 0.15
OBJECT_FOREGROUND_DEPTH_PERCENTILE = 35
OBJECT_FOREGROUND_RING_PIXELS = 10
OBJECT_FOREGROUND_MIN_VALID_PIXELS = 80
OBJECT_FOREGROUND_EXEMPT_LABELS = {'door', 'window', 'fan', 'light', 'mirror', 'picture', 'wall', 'floor', 'ceiling'}

# Final cleanup hard-rejects wall/floor/ceiling assignments that violate the same 3D height gates.
ENABLE_FINAL_STRUCTURAL_GEOMETRY_CLEANUP = True

# Approximate structural height gates derived from camera up direction and the target geometry.
# These prevent wall masks from labeling ceiling/floor bands and prevent floor/ceiling masks from
# spreading across vertical objects.
ENABLE_STRUCTURAL_HEIGHT_GATES = True
FLOOR_HEIGHT_PERCENTILE = 12
CEILING_HEIGHT_PERCENTILE = 88
STRUCTURAL_HEIGHT_MARGIN_M = 0.35

# Voting thresholds.
MIN_TOTAL_VOTE = 1.0
MIN_SUPPORT_VIEWS = 1
MIN_LABEL_MARGIN = 0.10

# Object grouping thresholds.
MIN_OBJECT_POINTS = 80
STRUCTURAL_LABELS = {'wall', 'floor', 'ceiling', 'door', 'window'}
DBSCAN_EPS_M = 0.35
DBSCAN_MIN_SAMPLES = 30
MAX_CLUSTER_POINTS_PER_LABEL = 80_000
MAX_PREVIEW_POINTS = 500_000

TRANSFORMS_CANDIDATES = [
    SCAN_DIR / 'transforms.json',
    SCAN_DIR / 'nerfstudio-data' / 'transforms.json',
]

SPLAT_CANDIDATES = [
    SCAN_DIR / 'exports' / 'splat.ply',
    SCAN_DIR / 'splat.ply',
    SCAN_DIR / 'gaussians' / 'splat.ply',
]

SPARSE_CANDIDATES = [
    SCAN_DIR / 'sparse_pc.ply',
    SCAN_DIR / 'nerfstudio-data' / 'sparse_pc.ply',
]

DEPTH_DIR_CANDIDATES = [
    SCAN_DIR / 'depth_variants' / 'median_depth',
    SCAN_DIR / 'depth_variants' / 'expected_depth',
    SCAN_DIR / 'depths',
    SCAN_DIR / 'depth',
]

OUTPUTS = {
    'labels_json': PROJECTION_DIR / 'splat_semantic_labels.json',
    'labeled_splat_ply': PROJECTION_DIR / 'splat_labeled.ply',
    'semantic_manifest_json': PROJECTION_DIR / 'semantic_scene_manifest.json',
    'semantic_manifest_csv': PROJECTION_DIR / 'semantic_scene_manifest.csv',
    'semantic_ply': PROJECTION_DIR / 'combined_objects_semantic_colored.ply',
    'bbox_json': PROJECTION_DIR / 'object_bounding_boxes.json',
    'bbox_ply': PROJECTION_DIR / 'object_bounding_boxes.ply',
    'summary_json': REPORT_DIR / 'geometry_labeling_summary.json',
    'frame_report_csv': REPORT_DIR / 'geometry_labeling_frame_report.csv',
    'label_summary_csv': REPORT_DIR / 'geometry_labeling_label_summary.csv',
    'coverage_report_json': REPORT_DIR / 'geometry_labeling_coverage_report.json',
    'role_competition_report_json': REPORT_DIR / 'geometry_labeling_role_competition_report.json',
    'vote_weight_report_csv': REPORT_DIR / 'geometry_labeling_vote_weight_report.csv',
    'mask_geometry_report_csv': REPORT_DIR / 'geometry_labeling_mask_geometry_report.csv',
    'footprint_vote_report_csv': REPORT_DIR / 'geometry_labeling_footprint_vote_report.csv',
}

for required in [SCAN_DIR, DETECTION_DIR]:
    if not required.exists():
        fail(f'Missing required folder: {required}')

logger.info('Project root: %s', PROJECT_ROOT)
logger.info('Scan dir: %s', SCAN_DIR)
logger.info('Detection dir: %s', DETECTION_DIR)
logger.info('Projection outputs: %s', PROJECTION_DIR)
logger.info('Log path: %s', LOG_PATH)
print('Pipeline version:', PIPELINE_VERSION)
logger.info('Pipeline version: %s', PIPELINE_VERSION)


## 4. Resolve Inputs

In [ ]:
# --- Cell explanation -----------------------------------------------------
# This cell resolves the core input files used by the rest of the notebook.
# It searches known Colab/Drive scan locations for transforms.json and the target geometry PLY.
# transforms.json supplies camera intrinsics/extrinsics for projecting 3D points into images.
# TARGET_PLY supplies the existing 3D points that will receive semantic labels.
# In the recommended mode, TARGET_POINT_SOURCE='splat' labels Gaussian centers from splat.ply.
# If a required file is missing, the cell fails immediately with the exact checked paths.
# ------------------------------------------------------------------------

def first_existing(candidates, label):
    for candidate in candidates:
        if candidate.exists():
            logger.info('%s: %s', label, candidate)
            return candidate
    checked = '\n'.join(str(p) for p in candidates)
    fail(f'Missing {label}. Checked:\n{checked}')

TRANSFORMS_PATH = first_existing(TRANSFORMS_CANDIDATES, 'transforms.json')

if TARGET_POINT_SOURCE == 'splat':
    TARGET_PLY = first_existing(SPLAT_CANDIDATES, 'target splat.ply')
elif TARGET_POINT_SOURCE == 'sparse':
    TARGET_PLY = first_existing(SPARSE_CANDIDATES, 'target sparse point cloud')
elif TARGET_POINT_SOURCE == 'auto':
    available = [p for p in SPLAT_CANDIDATES + SPARSE_CANDIDATES if p.exists()]
    if not available:
        fail('No target geometry PLY found for auto mode.')
    TARGET_PLY = available[0]
else:
    fail(f'Unknown TARGET_POINT_SOURCE: {TARGET_POINT_SOURCE}')

logger.info('Target geometry: %s', TARGET_PLY)


## 5. Load Cameras

In [ ]:
# --- Cell explanation -----------------------------------------------------
# This cell loads and normalizes camera information from transforms.json.
# It builds a frame_records dictionary keyed by stable frame ids like frame_000123.
# Each frame record stores:
# - the resolved RGB image path,
# - the camera-to-world matrix from Nerfstudio,
# - the inverse world-to-camera matrix used for projection,
# - the original frame metadata for intrinsics.
# It also builds frame_aliases so detection JSON filenames, image filenames, and transform entries
# can still match even if they use slightly different naming styles.
# ------------------------------------------------------------------------

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}

def normalize_key(value):
    stem = Path(str(value)).stem
    lower = stem.lower()
    match = re.search(r'(?:frame|image|img)[_\- ]*0*(\d+)$', lower)
    if match:
        return f'frame_{int(match.group(1)):06d}'
    if lower.isdigit():
        return f'frame_{int(lower):06d}'
    return lower

def resolve_relative_path(raw_path):
    if raw_path is None:
        return None
    path = Path(str(raw_path))
    candidates = []
    if path.is_absolute():
        candidates.append(path)
    else:
        candidates.extend([
            SCAN_DIR / path,
            SCAN_DIR / 'nerfstudio-data' / path,
            PROJECT_ROOT / path,
            path,
        ])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0] if candidates else None

def camera_to_world_matrix(frame):
    matrix = np.asarray(frame.get('transform_matrix'), dtype=np.float64)
    if matrix.shape != (4, 4):
        fail(f'Invalid transform_matrix shape for frame {frame.get("file_path")}: {matrix.shape}')
    return matrix

def frame_intrinsics(frame, transforms_data, image_shape):
    h, w = image_shape[:2]
    fx = frame.get('fl_x', transforms_data.get('fl_x'))
    fy = frame.get('fl_y', transforms_data.get('fl_y', fx))
    cx = frame.get('cx', transforms_data.get('cx', w / 2.0))
    cy = frame.get('cy', transforms_data.get('cy', h / 2.0))
    if fx is None or fy is None:
        camera_angle_x = frame.get('camera_angle_x', transforms_data.get('camera_angle_x'))
        if camera_angle_x is None:
            fail(f'Missing focal length/camera_angle_x for frame {frame.get("file_path")}')
        fx = 0.5 * w / math.tan(0.5 * float(camera_angle_x))
        fy = fx
    return float(fx), float(fy), float(cx), float(cy)

def image_shape_for_frame(frame):
    image_path = frame['image_path']
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        fail(f'Could not read frame image for shape: {image_path}')
    return image.shape[:2]

transforms_data = json.loads(TRANSFORMS_PATH.read_text())
frames = transforms_data.get('frames', [])
if not frames:
    fail(f'No frames found in transforms.json: {TRANSFORMS_PATH}')

frame_records = {}
frame_aliases = {}
missing_images = []

for frame_index, frame in enumerate(frames):
    raw_file_path = frame.get('file_path') or frame.get('filename') or frame.get('image_path')
    image_path = resolve_relative_path(raw_file_path)
    if image_path is None or not image_path.exists():
        missing_images.append(str(raw_file_path))
        continue
    key = normalize_key(image_path.name)
    c2w = camera_to_world_matrix(frame)
    record = {
        'frame_index': frame_index,
        'key': key,
        'image_path': image_path,
        'c2w': c2w,
        'w2c': np.linalg.inv(c2w),
        'frame': frame,
    }
    frame_records[key] = record
    for alias in {key, normalize_key(raw_file_path), normalize_key(image_path.name), image_path.stem.lower()}:
        frame_aliases[alias] = key

if not frame_records:
    sample_missing = '\n'.join(missing_images[:10])
    fail(f'No transform frames could be resolved to readable images. Sample missing:\n{sample_missing}')

logger.info('Resolved %d camera frames. Missing images: %d', len(frame_records), len(missing_images))
print('Resolved camera frames:', len(frame_records))
print('Missing frame images:', len(missing_images))
print('Example frame keys:', list(frame_records.keys())[:5])


## 6. Load 2D Detections, Masks, and Depth Index

In [ ]:
# --- Cell explanation -----------------------------------------------------
# This cell loads 2D detection/mask metadata and indexes rendered depth maps.
# It scans semantic_recon/detections/*.json, extracts object/layout masks, resolves each mask path,
# assigns a normalized label, and matches each mask to a camera frame from transforms.json.
# The resulting mask_df table is the bridge between 2D segmentation and 3D labeling.
# The depth_index maps frame keys to depth images, preferring the better depth_variants folders.
# Depth maps are used only for visibility/occlusion checks; they do not create new 3D geometry.
# A matched_masks.csv report is written so path/name mismatches can be debugged later.
# ------------------------------------------------------------------------

def mask_path_from_detection(det, json_path):
    for key in ['mask_path', 'segmentation_path', 'mask_file', 'path']:
        raw = det.get(key)
        if raw:
            p = Path(str(raw))
            candidates = []
            if p.is_absolute():
                candidates.append(p)
            else:
                candidates.extend([
                    SCAN_DIR / p,
                    SEM_DIR / p,
                    json_path.parent / p,
                    PROJECT_ROOT / p,
                    p,
                ])
            for candidate in candidates:
                if candidate.exists():
                    return candidate
            return candidates[0]
    return None

def detection_label(det):
    for key in ['label', 'class_name', 'name', 'category', 'class']:
        value = det.get(key)
        if value:
            return str(value).strip().lower().replace(' ', '_')
    return 'unknown'

def detection_score(det):
    for key in ['score', 'confidence', 'probability']:
        if key in det and det[key] is not None:
            try:
                return float(det[key])
            except Exception:
                pass
    return 1.0

def detection_lists(data):
    groups = []
    for key in ['detections', 'objects', 'instances']:
        if isinstance(data.get(key), list):
            groups.extend((key, item) for item in data[key])
    for key in ['layout_masks', 'layout', 'planes']:
        if isinstance(data.get(key), list):
            groups.extend((key, item) for item in data[key])
    if not groups and isinstance(data, list):
        groups.extend(('detections', item) for item in data)
    return groups

rows = []
json_paths = sorted(DETECTION_DIR.glob('*.json'))
if not json_paths:
    fail(f'No detection JSON files found under {DETECTION_DIR}. Run segmentation.ipynb first.')

for json_path in json_paths:
    try:
        data = json.loads(json_path.read_text())
    except Exception as exc:
        logger.warning('Skipping unreadable detection JSON %s: %s', json_path, exc)
        continue
    image_ref = data.get('image_path') or data.get('file_path') or data.get('frame_path') or json_path.stem
    frame_key = frame_aliases.get(normalize_key(image_ref), frame_aliases.get(normalize_key(json_path.stem)))
    if frame_key is None:
        continue
    for det_index, (source_group, det) in enumerate(detection_lists(data)):
        if not isinstance(det, dict):
            continue
        mask_path = mask_path_from_detection(det, json_path)
        if mask_path is None or not mask_path.exists():
            continue
        label = detection_label(det)
        role = 'structural' if label in STRUCTURAL_LABELS or source_group.startswith('layout') else 'small'
        rows.append({
            'frame_key': frame_key,
            'json_path': str(json_path),
            'mask_path': str(mask_path),
            'mask_id': f'{json_path.stem}_{det_index:03d}_{label}',
            'label': label,
            'role': role,
            'score': detection_score(det),
            'source_group': source_group,
        })

mask_df = pd.DataFrame(rows)
if mask_df.empty:
    fail('No usable masks matched camera frames. Check detection JSON mask_path values and transforms frame names.')

if MAX_MASKS_PER_FRAME is not None:
    mask_df = mask_df.groupby('frame_key', group_keys=False).head(MAX_MASKS_PER_FRAME).reset_index(drop=True)

def depth_key(path):
    return normalize_key(Path(path).stem)

def build_depth_index():
    depth_index = {}
    depth_source = {}
    for depth_dir in DEPTH_DIR_CANDIDATES:
        if not depth_dir.exists():
            continue
        for depth_path in sorted(depth_dir.iterdir()):
            if depth_path.suffix.lower() not in {'.npy', '.npz', '.png', '.tif', '.tiff', '.exr'}:
                continue
            key = depth_key(depth_path)
            if key not in depth_index:
                depth_index[key] = depth_path
                depth_source[key] = depth_dir.name
    return depth_index, depth_source

depth_index, depth_source = build_depth_index()

mask_df['has_depth'] = mask_df['frame_key'].map(lambda k: k in depth_index)

# Count how many masks each label has across the whole scan. Raw voting is biased toward labels
# with many masks, so each label gets an inverse-frequency weight before per-point votes are added.
label_mask_counts = mask_df['label'].value_counts().sort_index()
median_label_mask_count = float(np.median(label_mask_counts.values)) if len(label_mask_counts) else 1.0
label_frequency_weights = {}
for label, count in label_mask_counts.items():
    if ENABLE_LABEL_FREQUENCY_NORMALIZATION:
        raw_weight = (median_label_mask_count / max(float(count), 1.0)) ** LABEL_FREQUENCY_POWER
        weight = float(np.clip(raw_weight, LABEL_FREQUENCY_MIN_WEIGHT, LABEL_FREQUENCY_MAX_WEIGHT))
    else:
        weight = 1.0
    label_frequency_weights[label] = weight

mask_df['label_mask_count'] = mask_df['label'].map(label_mask_counts).astype(int)
mask_df['label_frequency_weight'] = mask_df['label'].map(label_frequency_weights).astype(float)
mask_df.to_csv(REPORT_DIR / 'matched_masks.csv', index=False)

vote_weight_report_df = (
    mask_df[['label', 'role', 'label_mask_count', 'label_frequency_weight']]
    .drop_duplicates('label')
    .sort_values(['role', 'label'])
)
vote_weight_report_df.to_csv(OUTPUTS['vote_weight_report_csv'], index=False)

print('Detection JSON files:', len(json_paths))
print('Usable masks:', len(mask_df))
print('Frames with masks:', mask_df['frame_key'].nunique())
print('Depth maps indexed:', len(depth_index))
print('Masks with depth:', int(mask_df['has_depth'].sum()), '/', len(mask_df))
print(mask_df.groupby(['role', 'label']).size().sort_values(ascending=False).head(20))
print('Label-frequency weights:')
display(vote_weight_report_df.head(30))

logger.info('Usable masks: %d across %d frames', len(mask_df), mask_df['frame_key'].nunique())
logger.info('Depth maps indexed: %d', len(depth_index))


## 7. Load Target Geometry

In [ ]:
# --- Cell explanation -----------------------------------------------------
# This cell reads the target PLY and extracts the XYZ coordinates that will be labeled.
# For Gaussian splats, these XYZ values are Gaussian centers, not a normal dense mesh surface.
# The cell keeps the original splat visual file untouched and works on label metadata/preview exports.
# It detects whether the PLY has Gaussian-specific fields, extracts approximate Gaussian footprint
# radii from scale_0..2, drops invalid points, optionally samples for faster testing, computes global
# bounds, and derives a world-up vector from camera poses.
# The world-up vector is used to build approximate floor/ceiling/wall height gates, which stops
# broad wall masks from claiming floor and ceiling bands.
# ------------------------------------------------------------------------

GAUSSIAN_FIELDS = {'f_dc_0', 'f_rest_0', 'opacity', 'scale_0', 'rot_0'}

def infer_gaussian_world_radius(vertex, names):
    scale_fields = ['scale_0', 'scale_1', 'scale_2']
    if not set(scale_fields).issubset(names):
        return np.full(len(vertex), GAUSSIAN_FOOTPRINT_FALLBACK_WORLD_RADIUS_M, dtype=np.float32), False
    scale_raw = np.column_stack([np.asarray(vertex[name], dtype=np.float32) for name in scale_fields])
    if GAUSSIAN_SCALE_IS_LOG:
        scale_values = np.exp(np.clip(scale_raw, -20.0, 10.0))
    else:
        scale_values = np.maximum(scale_raw, 1e-6)
    radius = np.max(scale_values, axis=1).astype(np.float32)
    valid = np.isfinite(radius) & (radius > 0)
    if not valid.any():
        return np.full(len(vertex), GAUSSIAN_FOOTPRINT_FALLBACK_WORLD_RADIUS_M, dtype=np.float32), False
    fallback = float(np.median(radius[valid]))
    radius = np.where(valid, radius, fallback).astype(np.float32)
    return radius, True

def load_ply_vertices(path):
    ply = PlyData.read(str(path))
    if 'vertex' not in ply:
        fail(f'PLY has no vertex element: {path}')
    vertex = ply['vertex'].data
    names = set(vertex.dtype.names or [])
    required = {'x', 'y', 'z'}
    if not required.issubset(names):
        fail(f'PLY is missing x/y/z vertex fields: {path}')
    points = np.column_stack([
        np.asarray(vertex['x'], dtype=np.float32),
        np.asarray(vertex['y'], dtype=np.float32),
        np.asarray(vertex['z'], dtype=np.float32),
    ])
    colors = None
    if {'red', 'green', 'blue'}.issubset(names):
        colors = np.column_stack([
            np.asarray(vertex['red'], dtype=np.uint8),
            np.asarray(vertex['green'], dtype=np.uint8),
            np.asarray(vertex['blue'], dtype=np.uint8),
        ])
    is_gaussian = bool(names & GAUSSIAN_FIELDS)
    gaussian_radius, has_gaussian_scale = infer_gaussian_world_radius(vertex, names)
    return points, colors, sorted(names), is_gaussian, gaussian_radius, has_gaussian_scale

points, source_colors, ply_fields, is_gaussian_splat, gaussian_world_radius, has_gaussian_scale = load_ply_vertices(TARGET_PLY)
finite_mask = np.isfinite(points).all(axis=1)
if not finite_mask.all():
    logger.warning('Dropping %d non-finite points from %s', int((~finite_mask).sum()), TARGET_PLY)
    points = points[finite_mask]
    if source_colors is not None:
        source_colors = source_colors[finite_mask]
    gaussian_world_radius = gaussian_world_radius[finite_mask]

if MAX_POINTS_TO_LABEL is not None and len(points) > MAX_POINTS_TO_LABEL:
    rng = np.random.default_rng(RANDOM_SEED)
    sample_indices = np.sort(rng.choice(len(points), size=MAX_POINTS_TO_LABEL, replace=False))
    points = points[sample_indices]
    if source_colors is not None:
        source_colors = source_colors[sample_indices]
    gaussian_world_radius = gaussian_world_radius[sample_indices]
else:
    sample_indices = None

if len(points) == 0:
    fail(f'No finite XYZ points loaded from {TARGET_PLY}')

bounds_min = points.min(axis=0)
bounds_max = points.max(axis=0)
summary = {
    'target_ply': str(TARGET_PLY),
    'target_point_source': TARGET_POINT_SOURCE,
    'point_count': int(len(points)),
    'is_gaussian_splat': bool(is_gaussian_splat),
    'fields': ply_fields,
    'bounds_min': bounds_min.tolist(),
    'bounds_max': bounds_max.tolist(),
    'bounds_size': (bounds_max - bounds_min).tolist(),
    'has_gaussian_scale': bool(has_gaussian_scale),
    'gaussian_radius_min': float(np.min(gaussian_world_radius)),
    'gaussian_radius_median': float(np.median(gaussian_world_radius)),
    'gaussian_radius_max': float(np.max(gaussian_world_radius)),
}

def derive_world_up_vector():
    camera_ups = []
    for record in frame_records.values():
        c2w = np.asarray(record['c2w'], dtype=np.float64)
        if c2w.shape == (4, 4):
            up = c2w[:3, 1]
            norm = np.linalg.norm(up)
            if np.isfinite(norm) and norm > 1e-6:
                camera_ups.append(up / norm)
    if not camera_ups:
        return np.array([0.0, 1.0, 0.0], dtype=np.float32)
    up = np.mean(np.vstack(camera_ups), axis=0)
    norm = np.linalg.norm(up)
    if not np.isfinite(norm) or norm < 1e-6:
        return np.array([0.0, 1.0, 0.0], dtype=np.float32)
    return (up / norm).astype(np.float32)

WORLD_UP_VECTOR = derive_world_up_vector()
point_heights = points @ WORLD_UP_VECTOR
floor_height_limit = float(np.percentile(point_heights, FLOOR_HEIGHT_PERCENTILE) + STRUCTURAL_HEIGHT_MARGIN_M)
ceiling_height_limit = float(np.percentile(point_heights, CEILING_HEIGHT_PERCENTILE) - STRUCTURAL_HEIGHT_MARGIN_M)
if floor_height_limit >= ceiling_height_limit:
    floor_height_limit = float(np.percentile(point_heights, 15))
    ceiling_height_limit = float(np.percentile(point_heights, 85))

summary.update({
    'world_up_vector': WORLD_UP_VECTOR.astype(float).tolist(),
    'height_min': float(point_heights.min()),
    'height_max': float(point_heights.max()),
    'floor_height_limit': floor_height_limit,
    'ceiling_height_limit': ceiling_height_limit,
})

print(json.dumps({k: v for k, v in summary.items() if k != 'fields'}, indent=2))
print('World up vector:', WORLD_UP_VECTOR)
print('Structural height gates:', {'floor_max': floor_height_limit, 'ceiling_min': ceiling_height_limit})
print('Gaussian footprint radius stats:', {
    'has_scale': bool(has_gaussian_scale),
    'min': float(np.min(gaussian_world_radius)),
    'median': float(np.median(gaussian_world_radius)),
    'max': float(np.max(gaussian_world_radius)),
})
logger.info('Loaded %d target points from %s. Gaussian fields: %s', len(points), TARGET_PLY, is_gaussian_splat)
logger.info('World up vector: %s; floor_height_limit=%.3f; ceiling_height_limit=%.3f', WORLD_UP_VECTOR, floor_height_limit, ceiling_height_limit)


## 8. Projection and Voting Helpers

In [ ]:
# --- Cell explanation -----------------------------------------------------
# This cell defines the reusable projection, visibility, mask, and PLY helper functions.
# Important pieces:
# - read_mask() loads a 2D mask and optionally dilates it by label role.
# - project_points() transforms world-space points into camera pixels using transforms.json.
# - rendered_depth_visible() accepts projected points that agree with rendered metric depth.
# - build_self_zbuffer() and self_zbuffer_visible() provide a geometry-based visibility fallback
#   for places where rendered depth is noisy or missing.
# - mask_dilation_for_label(), vote_multiplier_for_label(), label_frequency_weight_for_label(),
#   mask_area_weight(), and structural_geometry_gate() encode label-competition and vote
#   normalization rules that keep common/huge labels such as wall conservative.
# - gaussian_footprint_radius_pixels() and footprint_mask_fraction() approximate how much of a
#   Gaussian's rendered screen footprint overlaps a semantic mask.
# - object_foreground_geometry_supported() rejects object masks that only contain background
#   reconstruction geometry, such as bed masks landing on wall points when the bed is missing.
# - write_point_ply() and box_edges_from_min_max() export viewer/debug PLY files.
# ------------------------------------------------------------------------

def read_mask(mask_path, image_shape, dilation_pixels=0):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(f'Could not read mask: {mask_path}')
    target_h, target_w = image_shape[:2]
    if mask.shape[:2] != (target_h, target_w):
        mask = cv2.resize(mask, (target_w, target_h), interpolation=cv2.INTER_NEAREST)
    mask = (mask > 0).astype(np.uint8)
    if dilation_pixels and dilation_pixels > 0:
        kernel_size = 2 * int(dilation_pixels) + 1
        kernel = np.ones((kernel_size, kernel_size), dtype=np.uint8)
        mask = cv2.dilate(mask, kernel, iterations=1)
    return mask.astype(bool)

def read_depth(depth_path):
    suffix = depth_path.suffix.lower()
    if suffix == '.npy':
        depth = np.load(depth_path)
    elif suffix == '.npz':
        data = np.load(depth_path)
        if 'depth' in data:
            depth = data['depth']
        else:
            depth = data[data.files[0]]
    else:
        raw = cv2.imread(str(depth_path), cv2.IMREAD_UNCHANGED)
        if raw is None:
            raise FileNotFoundError(f'Could not read depth: {depth_path}')
        raw_dtype = raw.dtype
        depth = raw.astype(np.float32)
        if raw_dtype.kind in {'u', 'i'} and np.nanmax(depth) > 255:
            # Common 16-bit export convention: millimeters.
            depth = depth / 1000.0
    depth = np.asarray(depth, dtype=np.float32)
    if depth.ndim == 3:
        depth = depth[..., 0]
    return depth


# Project one chunk of world-space XYZ points into a single camera image.
# Chunking keeps memory bounded when splat.ply contains hundreds of thousands of Gaussians.
def project_points(points_chunk, record, image_shape):
    h, w = image_shape[:2]
    fx, fy, cx, cy = frame_intrinsics(record['frame'], transforms_data, image_shape)
    homo = np.ones((len(points_chunk), 4), dtype=np.float64)
    homo[:, :3] = points_chunk.astype(np.float64)
    cam = (record['w2c'] @ homo.T).T[:, :3]

    if CAMERA_CONVENTION == 'nerfstudio_opengl':
        # Nerfstudio/Blender camera space: +X right, +Y up, -Z forward.
        depth_z = -cam[:, 2]
        image_x = cam[:, 0]
        image_y = -cam[:, 1]
    elif CAMERA_CONVENTION == 'opencv':
        # OpenCV/COLMAP camera space: +X right, +Y down, +Z forward.
        depth_z = cam[:, 2]
        image_x = cam[:, 0]
        image_y = cam[:, 1]
    else:
        fail(f'Unknown CAMERA_CONVENTION: {CAMERA_CONVENTION}')

    valid = np.isfinite(depth_z) & (depth_z > MIN_DEPTH_M) & (depth_z < MAX_DEPTH_M)
    u_float = np.full_like(depth_z, np.nan, dtype=np.float64)
    v_float = np.full_like(depth_z, np.nan, dtype=np.float64)
    u_float[valid] = fx * (image_x[valid] / depth_z[valid]) + cx
    v_float[valid] = fy * (image_y[valid] / depth_z[valid]) + cy
    valid &= np.isfinite(u_float) & np.isfinite(v_float)
    valid &= (u_float >= 0) & (u_float < w) & (v_float >= 0) & (v_float < h)
    idx = np.flatnonzero(valid)
    u = np.floor(u_float[idx]).astype(np.int32)
    v = np.floor(v_float[idx]).astype(np.int32)
    return idx, u, v, depth_z[idx].astype(np.float32)


# Build a per-frame nearest-depth image from the target geometry itself.
# This acts as a fallback visibility test when rendered depth maps miss valid room surfaces.
def build_self_zbuffer(record, image_shape):
    h, w = image_shape[:2]
    large = np.float32(np.inf)
    zbuffer = np.full((h, w), large, dtype=np.float32)
    projected = 0
    for start in range(0, len(points), POINT_CHUNK_SIZE):
        end = min(start + POINT_CHUNK_SIZE, len(points))
        local_idx, u, v, z = project_points(points[start:end], record, image_shape)
        if len(local_idx) == 0:
            continue
        projected += len(local_idx)
        np.minimum.at(zbuffer, (v, u), z)
    if SELF_ZBUFFER_RADIUS_PX and SELF_ZBUFFER_RADIUS_PX > 0:
        kernel_size = 2 * int(SELF_ZBUFFER_RADIUS_PX) + 1
        kernel = np.ones((kernel_size, kernel_size), dtype=np.uint8)
        zbuffer = cv2.erode(zbuffer, kernel, iterations=1)
    valid_pixels = int(np.isfinite(zbuffer).sum())
    return zbuffer, projected, valid_pixels

def gaussian_footprint_sample_offsets():
    if GAUSSIAN_FOOTPRINT_SAMPLE_PATTERN == 'center':
        offsets = [(0.0, 0.0, 1.0)]
    elif GAUSSIAN_FOOTPRINT_SAMPLE_PATTERN == 'center_4':
        offsets = [
            (0.0, 0.0, 1.0),
            (0.55, 0.0, 0.65), (-0.55, 0.0, 0.65),
            (0.0, 0.55, 0.65), (0.0, -0.55, 0.65),
        ]
    else:
        offsets = [
            (0.0, 0.0, 1.0),
            (0.50, 0.0, 0.70), (-0.50, 0.0, 0.70),
            (0.0, 0.50, 0.70), (0.0, -0.50, 0.70),
            (0.45, 0.45, 0.50), (-0.45, 0.45, 0.50),
            (0.45, -0.45, 0.50), (-0.45, -0.45, 0.50),
        ]
    dx = np.asarray([item[0] for item in offsets], dtype=np.float32)
    dy = np.asarray([item[1] for item in offsets], dtype=np.float32)
    weights = np.asarray([item[2] for item in offsets], dtype=np.float32)
    weights = weights / np.maximum(weights.sum(), 1e-6)
    return dx, dy, weights

FOOTPRINT_DX, FOOTPRINT_DY, FOOTPRINT_WEIGHTS = gaussian_footprint_sample_offsets()

def gaussian_footprint_radius_pixels(point_indices, depth_z, record, image_shape):
    if not ENABLE_GAUSSIAN_FOOTPRINT_VOTING:
        return np.ones(len(point_indices), dtype=np.float32) * GAUSSIAN_FOOTPRINT_MIN_RADIUS_PX
    fx, fy, _, _ = frame_intrinsics(record['frame'], transforms_data, image_shape)
    focal = max(float(fx), float(fy))
    world_radius = gaussian_world_radius[point_indices]
    radius_px = GAUSSIAN_FOOTPRINT_STDDEV_MULTIPLIER * focal * world_radius / np.maximum(depth_z, 1e-6)
    radius_px = np.where(np.isfinite(radius_px), radius_px, GAUSSIAN_FOOTPRINT_MIN_RADIUS_PX)
    radius_px = np.clip(radius_px, GAUSSIAN_FOOTPRINT_MIN_RADIUS_PX, GAUSSIAN_FOOTPRINT_MAX_RADIUS_PX)
    return radius_px.astype(np.float32)

def footprint_mask_fraction(mask, u, v, radius_px):
    if len(u) == 0:
        return np.zeros(0, dtype=np.float32)
    h, w = mask.shape[:2]
    coverage = np.zeros(len(u), dtype=np.float32)
    total_weight = np.float32(0.0)
    for dx, dy, weight in zip(FOOTPRINT_DX, FOOTPRINT_DY, FOOTPRINT_WEIGHTS):
        su = np.rint(u.astype(np.float32) + dx * radius_px).astype(np.int32)
        sv = np.rint(v.astype(np.float32) + dy * radius_px).astype(np.int32)
        valid = (su >= 0) & (su < w) & (sv >= 0) & (sv < h)
        sample_hit = np.zeros(len(u), dtype=np.float32)
        if valid.any():
            sample_hit[valid] = mask[sv[valid], su[valid]].astype(np.float32)
        coverage += float(weight) * sample_hit
        total_weight += np.float32(weight)
    return coverage / max(float(total_weight), 1e-6)

def rendered_depth_visible(projected_u, projected_v, projected_z, depth, image_shape):
    image_h, image_w = image_shape[:2]
    depth_h, depth_w = depth.shape[:2]
    du0 = np.floor(projected_u.astype(np.float64) * depth_w / image_w).astype(np.int32)
    dv0 = np.floor(projected_v.astype(np.float64) * depth_h / image_h).astype(np.int32)
    best_delta = np.full(len(projected_z), np.inf, dtype=np.float32)
    best_ref = np.full(len(projected_z), np.nan, dtype=np.float32)

    radius = int(max(0, DEPTH_PATCH_RADIUS_PX))
    for oy in range(-radius, radius + 1):
        for ox in range(-radius, radius + 1):
            du = np.clip(du0 + ox, 0, depth_w - 1)
            dv = np.clip(dv0 + oy, 0, depth_h - 1)
            reference_z = depth[dv, du]
            valid_depth = np.isfinite(reference_z) & (reference_z > MIN_DEPTH_M) & (reference_z < MAX_DEPTH_M)
            delta = np.abs(projected_z - reference_z)
            better = valid_depth & (delta < best_delta)
            best_delta[better] = delta[better]
            best_ref[better] = reference_z[better]

    tolerance = np.maximum(DEPTH_ABS_TOLERANCE_M, DEPTH_REL_TOLERANCE * best_ref)
    visible = np.isfinite(best_ref) & (best_delta <= tolerance)
    return visible, best_delta, best_ref

def self_zbuffer_visible(projected_u, projected_v, projected_z, zbuffer):
    reference_z = zbuffer[projected_v, projected_u]
    tolerance = np.maximum(SELF_ZBUFFER_ABS_TOLERANCE_M, SELF_ZBUFFER_REL_TOLERANCE * reference_z)
    visible = np.isfinite(reference_z) & (np.abs(projected_z - reference_z) <= tolerance)
    return visible, reference_z


# Combine rendered-depth visibility and self-zbuffer visibility into one boolean mask.
# Hybrid mode gives full-weight votes to rendered-depth matches and lower-weight votes to zbuffer matches.
def visibility_for_projection(u, v, z, depth, zbuffer, image_shape):
    if VISIBILITY_MODE == 'none':
        return np.ones(len(z), dtype=bool), np.full(len(z), NO_DEPTH_VOTE_WEIGHT, dtype=np.float32), 'none', 0, 0

    rendered_visible = np.zeros(len(z), dtype=bool)
    rendered_count = 0
    if depth is not None and VISIBILITY_MODE in {'hybrid', 'rendered_depth'}:
        rendered_visible, _, _ = rendered_depth_visible(u, v, z, depth, image_shape)
        rendered_count = int(rendered_visible.sum())

    self_visible = np.zeros(len(z), dtype=bool)
    self_count = 0
    if zbuffer is not None and VISIBILITY_MODE in {'hybrid', 'self_zbuffer'}:
        self_visible, _ = self_zbuffer_visible(u, v, z, zbuffer)
        self_count = int(self_visible.sum())

    if VISIBILITY_MODE == 'rendered_depth':
        visible = rendered_visible
        weights = np.where(rendered_visible, RENDERED_DEPTH_VOTE_WEIGHT, 0.0).astype(np.float32)
        source = 'rendered_depth'
    elif VISIBILITY_MODE == 'self_zbuffer':
        visible = self_visible
        weights = np.where(self_visible, SELF_ZBUFFER_VOTE_WEIGHT, 0.0).astype(np.float32)
        source = 'self_zbuffer'
    elif VISIBILITY_MODE == 'hybrid':
        visible = rendered_visible | self_visible
        weights = np.zeros(len(z), dtype=np.float32)
        weights[self_visible] = SELF_ZBUFFER_VOTE_WEIGHT
        weights[rendered_visible] = RENDERED_DEPTH_VOTE_WEIGHT
        source = 'hybrid'
    else:
        fail(f'Unknown VISIBILITY_MODE: {VISIBILITY_MODE}')

    if not visible.any() and depth is None and ALLOW_NO_DEPTH_FRAME_VOTES:
        visible = np.ones(len(z), dtype=bool)
        weights = np.full(len(z), NO_DEPTH_VOTE_WEIGHT, dtype=np.float32)
        source = 'no_depth_low_weight'

    return visible, weights, source, rendered_count, self_count

def mask_dilation_for_label(label, role):
    if label == 'wall':
        return WALL_MASK_DILATION_PIXELS
    if label in SURFACE_STRUCTURAL_LABELS:
        return STRUCTURAL_MASK_DILATION_PIXELS
    return SMALL_MASK_DILATION_PIXELS

def vote_multiplier_for_label(label, role):
    if label not in SURFACE_STRUCTURAL_LABELS:
        return SMALL_OBJECT_VOTE_MULTIPLIER
    return STRUCTURAL_VOTE_MULTIPLIERS.get(label, DEFAULT_STRUCTURAL_VOTE_MULTIPLIER)


# Apply coarse 3D height rules to broad surface labels.
# This prevents wall from claiming low floor points or high ceiling points even when 2D masks overlap.
def label_frequency_weight_for_label(label):
    return float(label_frequency_weights.get(label, 1.0))

def mask_area_weight(mask):
    if not ENABLE_MASK_AREA_NORMALIZATION:
        return 1.0, float(mask.mean())
    area_fraction = float(mask.mean())
    safe_area = max(area_fraction, 1e-6)
    raw_weight = (MASK_AREA_REFERENCE_FRACTION / safe_area) ** MASK_AREA_POWER
    weight = float(np.clip(raw_weight, MASK_AREA_MIN_WEIGHT, MASK_AREA_MAX_WEIGHT))
    return weight, area_fraction

def structural_geometry_gate(label, point_indices):
    if not ENABLE_STRUCTURAL_HEIGHT_GATES or label not in SURFACE_STRUCTURAL_LABELS:
        return np.ones(len(point_indices), dtype=bool)
    heights = point_heights[point_indices]
    if label == 'floor':
        return heights <= floor_height_limit
    if label == 'ceiling':
        return heights >= ceiling_height_limit
    if label == 'wall':
        return (heights > floor_height_limit) & (heights < ceiling_height_limit)
    return np.ones(len(point_indices), dtype=bool)

def dilate_bool_mask(mask, pixels):
    if not pixels or pixels <= 0:
        return mask
    kernel_size = 2 * int(pixels) + 1
    kernel = np.ones((kernel_size, kernel_size), dtype=np.uint8)
    return cv2.dilate(mask.astype(np.uint8), kernel, iterations=1).astype(bool)

def depth_image_to_image_shape(depth, image_shape):
    if depth is None:
        return None
    target_h, target_w = image_shape[:2]
    if depth.shape[:2] == (target_h, target_w):
        return depth
    return cv2.resize(depth.astype(np.float32), (target_w, target_h), interpolation=cv2.INTER_NEAREST)

def foreground_check_required(label):
    if not ENABLE_OBJECT_FOREGROUND_GEOMETRY_CHECK:
        return False
    return label not in OBJECT_FOREGROUND_EXEMPT_LABELS and label not in SURFACE_STRUCTURAL_LABELS

def object_foreground_geometry_supported(mask, geometry_depth_image):
    if geometry_depth_image is None:
        return True, {
            'geometry_supported': True,
            'foreground_check_reason': 'no_geometry_depth_available',
            'inside_valid_pixels': 0,
            'ring_valid_pixels': 0,
            'foreground_delta_m': None,
        }

    valid_depth = np.isfinite(geometry_depth_image) & (geometry_depth_image > MIN_DEPTH_M) & (geometry_depth_image < MAX_DEPTH_M)
    ring = dilate_bool_mask(mask, OBJECT_FOREGROUND_RING_PIXELS) & ~mask
    inside_values = geometry_depth_image[mask & valid_depth]
    ring_values = geometry_depth_image[ring & valid_depth]

    inside_count = int(len(inside_values))
    ring_count = int(len(ring_values))
    if inside_count < OBJECT_FOREGROUND_MIN_VALID_PIXELS or ring_count < OBJECT_FOREGROUND_MIN_VALID_PIXELS:
        return True, {
            'geometry_supported': True,
            'foreground_check_reason': 'insufficient_valid_depth_for_check',
            'inside_valid_pixels': inside_count,
            'ring_valid_pixels': ring_count,
            'foreground_delta_m': None,
        }

    inside_depth = float(np.percentile(inside_values, OBJECT_FOREGROUND_DEPTH_PERCENTILE))
    ring_depth = float(np.median(ring_values))
    foreground_delta = ring_depth - inside_depth
    supported = bool(foreground_delta >= OBJECT_FOREGROUND_MIN_DELTA_M)
    return supported, {
        'geometry_supported': supported,
        'foreground_check_reason': 'foreground_depth_delta',
        'inside_valid_pixels': inside_count,
        'ring_valid_pixels': ring_count,
        'inside_depth_m': inside_depth,
        'ring_depth_m': ring_depth,
        'foreground_delta_m': foreground_delta,
    }

def write_augmented_gaussian_ply(src_path, dst_path, label_ids, label_confidence, support_views):
    if sample_indices is not None:
        logger.warning('Skipping augmented splat PLY because MAX_POINTS_TO_LABEL sampled the point set.')
        return False
    ply = PlyData.read(str(src_path))
    vertex = ply['vertex'].data
    if len(vertex) != len(label_ids):
        logger.warning('Skipping augmented splat PLY: source vertex count %d != label count %d', len(vertex), len(label_ids))
        return False
    extra_dtype = [
        ('semantic_label_id', 'i4'),
        ('semantic_confidence', 'f4'),
        ('semantic_support_views', 'u2'),
    ]
    new_dtype = vertex.dtype.descr + extra_dtype
    augmented = np.empty(vertex.shape, dtype=new_dtype)
    for name in vertex.dtype.names:
        augmented[name] = vertex[name]
    augmented['semantic_label_id'] = np.asarray(label_ids, dtype=np.int32)
    augmented['semantic_confidence'] = np.asarray(label_confidence, dtype=np.float32)
    augmented['semantic_support_views'] = np.asarray(support_views, dtype=np.uint16)

    elements = []
    for element in ply.elements:
        if element.name == 'vertex':
            elements.append(PlyElement.describe(augmented, 'vertex'))
        else:
            elements.append(element)
    PlyData(elements, text=False).write(str(dst_path))
    return True

def stable_label_color(label):
    import hashlib
    digest = hashlib.sha1(label.encode('utf-8')).digest()
    base = np.array([digest[0], digest[1], digest[2]], dtype=np.uint8)
    return np.maximum(base, np.array([55, 55, 55], dtype=np.uint8))

def write_point_ply(path, xyz, rgb):
    xyz = np.asarray(xyz, dtype=np.float32)
    rgb = np.asarray(rgb, dtype=np.uint8)
    vertex = np.empty(len(xyz), dtype=[('x', 'f4'), ('y', 'f4'), ('z', 'f4'), ('red', 'u1'), ('green', 'u1'), ('blue', 'u1')])
    vertex['x'] = xyz[:, 0]
    vertex['y'] = xyz[:, 1]
    vertex['z'] = xyz[:, 2]
    vertex['red'] = rgb[:, 0]
    vertex['green'] = rgb[:, 1]
    vertex['blue'] = rgb[:, 2]
    PlyData([PlyElement.describe(vertex, 'vertex')], text=False).write(str(path))

def box_edges_from_min_max(min_xyz, max_xyz):
    x0, y0, z0 = min_xyz
    x1, y1, z1 = max_xyz
    corners = np.array([
        [x0, y0, z0], [x1, y0, z0], [x1, y1, z0], [x0, y1, z0],
        [x0, y0, z1], [x1, y0, z1], [x1, y1, z1], [x0, y1, z1],
    ], dtype=np.float32)
    edges = [(0,1), (1,2), (2,3), (3,0), (4,5), (5,6), (6,7), (7,4), (0,4), (1,5), (2,6), (3,7)]
    samples = []
    for a, b in edges:
        for t in np.linspace(0, 1, 12):
            samples.append(corners[a] * (1 - t) + corners[b] * t)
    return np.asarray(samples, dtype=np.float32)


## 9. Label Existing Geometry by Projecting It Into Masks

In [ ]:
# --- Cell explanation -----------------------------------------------------
# This is the main 3D semantic voting cell.
# For each frame with masks, it projects the existing 3D target points into that frame, filters
# projected points by visibility, samples each visible Gaussian's approximate screen footprint
# against masks, and accumulates weighted label votes per Gaussian. Votes are normalized by label frequency and mask area,
# so common/huge labels such as wall do not win simply by appearing often or covering many pixels.
# Object-like labels vote first and suppress broad surface labels in the same image region.
# Surface labels wall/floor/ceiling vote only after object masks and only if their height gates pass.
# The outputs of this cell are label_scores, label_views, total_votes, and frame-level reports.
# These arrays are not final labels yet; they are raw accumulated evidence for the next cell.
# ------------------------------------------------------------------------

labels_seen = sorted(mask_df['label'].dropna().unique().tolist())
label_scores = {label: np.zeros(len(points), dtype=np.float32) for label in labels_seen}
label_views = {label: np.zeros(len(points), dtype=np.uint16) for label in labels_seen}
total_votes = np.zeros(len(points), dtype=np.float32)
any_support_views = np.zeros(len(points), dtype=np.uint16)
frame_reports = []
role_competition_stats = defaultdict(int)
vote_weight_rows = []
mask_geometry_rows = []
footprint_vote_rows = []

frame_groups = list(mask_df.groupby('frame_key'))
if MAX_FRAMES_TO_PROCESS is not None:
    frame_groups = frame_groups[:MAX_FRAMES_TO_PROCESS]


# Iterate frame-by-frame so every 2D mask votes only through its matching camera pose.
for frame_key, group in tqdm(frame_groups, desc='Voting labels from masks'):
    record = frame_records.get(frame_key)
    if record is None:
        frame_reports.append({'frame_key': frame_key, 'status': 'missing_camera', 'masks': len(group), 'votes': 0})
        continue

    image_bgr = cv2.imread(str(record['image_path']), cv2.IMREAD_COLOR)
    if image_bgr is None:
        frame_reports.append({'frame_key': frame_key, 'status': 'unreadable_image', 'masks': len(group), 'votes': 0})
        continue
    image_shape = image_bgr.shape[:2]

    depth = None
    if frame_key in depth_index:
        try:
            depth = read_depth(depth_index[frame_key])
        except Exception as exc:
            logger.warning('Could not read depth for %s: %s', frame_key, exc)
            depth = None

    zbuffer = None
    zbuffer_projected = 0
    zbuffer_pixels = 0
    if VISIBILITY_MODE in {'hybrid', 'self_zbuffer'}:
        zbuffer, zbuffer_projected, zbuffer_pixels = build_self_zbuffer(record, image_shape)

    if VISIBILITY_MODE == 'rendered_depth' and depth is None and not ALLOW_NO_DEPTH_FRAME_VOTES:
        frame_reports.append({'frame_key': frame_key, 'status': 'skipped_no_depth', 'masks': len(group), 'votes': 0})
        continue

    prepared_masks = []
    for row in group.itertuples(index=False):
        label = row.label
        role = row.role
        dilation_pixels = mask_dilation_for_label(label, role)
        try:
            mask = read_mask(Path(row.mask_path), image_shape, dilation_pixels=dilation_pixels)
        except Exception as exc:
            logger.warning('Skipping unreadable mask %s: %s', row.mask_path, exc)
            continue
        if mask.sum() == 0:
            continue
        frequency_weight = label_frequency_weight_for_label(label)
        area_weight, area_fraction = mask_area_weight(mask)
        role_weight = vote_multiplier_for_label(label, role)
        combined_static_weight = frequency_weight * area_weight * role_weight
        prepared_masks.append({
            'mask': mask,
            'label': label,
            'score': max(float(row.score), 0.05),
            'role': role,
            'mask_id': row.mask_id,
            'vote_multiplier': role_weight,
            'label_frequency_weight': frequency_weight,
            'mask_area_weight': area_weight,
            'mask_area_fraction': area_fraction,
            'static_vote_weight': combined_static_weight,
        })
        vote_weight_rows.append({
            'frame_key': frame_key,
            'mask_id': row.mask_id,
            'label': label,
            'role': role,
            'score': max(float(row.score), 0.05),
            'label_mask_count': int(label_mask_counts.get(label, 0)),
            'label_frequency_weight': frequency_weight,
            'mask_area_fraction': area_fraction,
            'mask_area_weight': area_weight,
            'role_weight': role_weight,
            'static_vote_weight': combined_static_weight,
        })

    if not prepared_masks:
        frame_reports.append({'frame_key': frame_key, 'status': 'no_readable_masks', 'masks': len(group), 'votes': 0})
        continue

    object_masks = [item for item in prepared_masks if item['label'] not in SURFACE_STRUCTURAL_LABELS]
    structural_masks = [item for item in prepared_masks if item['label'] in SURFACE_STRUCTURAL_LABELS]

    geometry_depth_image = zbuffer if zbuffer is not None else depth_image_to_image_shape(depth, image_shape)
    geometry_supported_object_masks = []
    for mask_rec in object_masks:
        supported = True
        stats = {
            'geometry_supported': True,
            'foreground_check_reason': 'not_required_for_label',
            'inside_valid_pixels': 0,
            'ring_valid_pixels': 0,
            'foreground_delta_m': None,
        }
        if foreground_check_required(mask_rec['label']):
            supported, stats = object_foreground_geometry_supported(mask_rec['mask'], geometry_depth_image)
        mask_geometry_rows.append({
            'frame_key': frame_key,
            'mask_id': mask_rec['mask_id'],
            'label': mask_rec['label'],
            'role': mask_rec['role'],
            **stats,
        })
        if supported:
            geometry_supported_object_masks.append(mask_rec)
        else:
            role_competition_stats[f'object_masks_rejected_foreground_{mask_rec["label"]}'] += 1
    object_masks = geometry_supported_object_masks

    # Small-object masks suppress structural votes in the same image region. This is the main
    # protection against giant wall masks consuming beds, desks, pillows, fans, doors, etc.

    # Build a union of object-like masks. The dilated union removes those image regions from
    # wall/floor/ceiling masks so broad structural masks cannot overwrite object evidence.
    if object_masks:
        small_union = np.zeros(image_shape, dtype=bool)

        # Object-like labels claim visible points first. A claimed point cannot receive a
        # wall/floor/ceiling vote from the same frame.
        for mask_rec in object_masks:
            small_union |= mask_rec['mask']
        small_suppression = dilate_bool_mask(small_union, SMALL_OBJECT_SUPPRESSION_DILATION_PIXELS)

        # Broad surface labels vote after object masks and only for unclaimed points.
        # The structural_geometry_gate() check further separates floor, wall, and ceiling by height.
        for mask_rec in structural_masks:
            before = int(mask_rec['mask'].sum())
            mask_rec['mask'] = mask_rec['mask'] & ~small_suppression
            role_competition_stats[f'structural_pixels_suppressed_{mask_rec["label"]}'] += before - int(mask_rec['mask'].sum())

    # Highest-confidence object masks claim points first. Structural masks only get points
    # that were not claimed by small objects in that frame.
    object_masks.sort(key=lambda item: (item['score'] * item['static_vote_weight']), reverse=True)
    structural_masks.sort(key=lambda item: (item['static_vote_weight'], item['score']), reverse=True)

    frame_vote_count = 0
    frame_projected = 0
    frame_visible = 0
    frame_rendered_visible = 0
    frame_self_visible = 0
    frame_mask_hits = 0
    frame_small_votes = 0
    frame_structural_votes = 0
    frame_structural_gate_rejections = 0

    for start in range(0, len(points), POINT_CHUNK_SIZE):
        end = min(start + POINT_CHUNK_SIZE, len(points))
        local_idx, u, v, z = project_points(points[start:end], record, image_shape)
        if len(local_idx) == 0:
            continue
        frame_projected += len(local_idx)
        global_idx = local_idx + start

        visible, visibility_weights, visibility_source, rendered_count, self_count = visibility_for_projection(u, v, z, depth, zbuffer, image_shape)
        frame_rendered_visible += rendered_count
        frame_self_visible += self_count
        if not visible.any():
            continue

        global_idx = global_idx[visible]
        u = u[visible]
        v = v[visible]
        z_visible = z[visible]
        visibility_weights = visibility_weights[visible]
        footprint_radius_px = gaussian_footprint_radius_pixels(global_idx, z_visible, record, image_shape)
        frame_visible += len(global_idx)

        claimed = np.zeros(len(global_idx), dtype=bool)
        voted_this_frame = np.zeros(len(global_idx), dtype=bool)

        for mask_rec in object_masks:
            coverage = footprint_mask_fraction(mask_rec['mask'], u, v, footprint_radius_px)
            inside = (coverage >= GAUSSIAN_FOOTPRINT_MIN_MASK_FRACTION) & ~claimed
            if not inside.any():
                continue
            ids = global_idx[inside]
            label = mask_rec['label']
            # Static vote weight = role/class prior * inverse label frequency * inverse mask area.
            # Footprint coverage approximates how much of this Gaussian's rendered footprint overlaps
            # the semantic mask, so a partial overlap gets a fractional vote instead of a binary hit.
            vote_weights = (
                float(mask_rec['score'])
                * float(mask_rec['static_vote_weight'])
                * coverage[inside]
                * visibility_weights[inside]
            )
            label_scores[label][ids] += vote_weights.astype(np.float32)
            label_views[label][ids] += 1
            total_votes[ids] += vote_weights.astype(np.float32)
            claimed[inside] = True
            voted_this_frame[inside] = True
            hit_count = int(inside.sum())
            if label in GAUSSIAN_FOOTPRINT_REPORT_LABELS:
                footprint_vote_rows.append({
                    'frame_key': frame_key,
                    'label': label,
                    'mask_id': mask_rec['mask_id'],
                    'role': mask_rec['role'],
                    'accepted_gaussians': hit_count,
                    'mean_footprint_radius_px': float(np.mean(footprint_radius_px[inside])),
                    'mean_mask_coverage': float(np.mean(coverage[inside])),
                    'mean_vote_weight': float(np.mean(vote_weights)),
                })
            frame_small_votes += hit_count
            frame_mask_hits += hit_count
            frame_vote_count += hit_count

        for mask_rec in structural_masks:
            coverage = footprint_mask_fraction(mask_rec['mask'], u, v, footprint_radius_px)
            inside = (coverage >= GAUSSIAN_FOOTPRINT_MIN_MASK_FRACTION) & ~claimed
            if not inside.any():
                continue
            candidate_ids = global_idx[inside]
            gate = structural_geometry_gate(mask_rec['label'], candidate_ids)
            if not gate.any():
                frame_structural_gate_rejections += int(inside.sum())
                continue
            rejected = int((~gate).sum())
            frame_structural_gate_rejections += rejected
            inside_positions = np.flatnonzero(inside)
            visible_positions = inside_positions[gate]
            ids = global_idx[visible_positions]
            label = mask_rec['label']
            # Surface labels use the same normalized vote formula, but only after object labels
            # have claimed their points and after the 3D structural height gate passes.
            vote_weights = (
                float(mask_rec['score'])
                * float(mask_rec['static_vote_weight'])
                * coverage[visible_positions]
                * visibility_weights[visible_positions]
            )
            label_scores[label][ids] += vote_weights.astype(np.float32)
            label_views[label][ids] += 1
            total_votes[ids] += vote_weights.astype(np.float32)
            claimed[visible_positions] = True
            voted_this_frame[visible_positions] = True
            hit_count = int(len(ids))
            if label in GAUSSIAN_FOOTPRINT_REPORT_LABELS:
                footprint_vote_rows.append({
                    'frame_key': frame_key,
                    'label': label,
                    'mask_id': mask_rec['mask_id'],
                    'role': mask_rec['role'],
                    'accepted_gaussians': hit_count,
                    'mean_footprint_radius_px': float(np.mean(footprint_radius_px[visible_positions])),
                    'mean_mask_coverage': float(np.mean(coverage[visible_positions])),
                    'mean_vote_weight': float(np.mean(vote_weights)),
                })
            frame_structural_votes += hit_count
            frame_mask_hits += hit_count
            frame_vote_count += hit_count

        any_support_views[global_idx[voted_this_frame]] += 1

    role_competition_stats['small_votes'] += frame_small_votes
    role_competition_stats['structural_votes'] += frame_structural_votes
    role_competition_stats['structural_gate_rejections'] += frame_structural_gate_rejections

    frame_reports.append({
        'frame_key': frame_key,
        'status': 'processed',
        'visibility_mode': VISIBILITY_MODE,
        'masks': len(prepared_masks),
        'object_masks': len(object_masks),
        'structural_masks': len(structural_masks),
        'projected_points': int(frame_projected),
        'visible_points': int(frame_visible),
        'rendered_depth_visible_points': int(frame_rendered_visible),
        'self_zbuffer_visible_points': int(frame_self_visible),
        'self_zbuffer_projected_points': int(zbuffer_projected),
        'self_zbuffer_valid_pixels': int(zbuffer_pixels),
        'mask_hits': int(frame_mask_hits),
        'small_votes': int(frame_small_votes),
        'structural_votes': int(frame_structural_votes),
        'structural_gate_rejections': int(frame_structural_gate_rejections),
        'votes': int(frame_vote_count),
        'depth_path': str(depth_index.get(frame_key, '')),
    })

frame_report_df = pd.DataFrame(frame_reports)
frame_report_df.to_csv(OUTPUTS['frame_report_csv'], index=False)
if vote_weight_rows:
    vote_weight_report_df = pd.DataFrame(vote_weight_rows).sort_values(['role', 'label', 'frame_key', 'mask_id'])
    vote_weight_report_df.to_csv(OUTPUTS['vote_weight_report_csv'], index=False)
if mask_geometry_rows:
    mask_geometry_report_df = pd.DataFrame(mask_geometry_rows).sort_values(['label', 'frame_key', 'mask_id'])
    mask_geometry_report_df.to_csv(OUTPUTS['mask_geometry_report_csv'], index=False)
footprint_vote_columns = [
    'frame_key', 'label', 'mask_id', 'role',
    'accepted_gaussians', 'mean_footprint_radius_px', 'mean_mask_coverage', 'mean_vote_weight',
]
if footprint_vote_rows:
    footprint_vote_report_df = pd.DataFrame(footprint_vote_rows).sort_values(['label', 'frame_key', 'mask_id'])
else:
    footprint_vote_report_df = pd.DataFrame(columns=footprint_vote_columns)
footprint_vote_report_df.to_csv(OUTPUTS['footprint_vote_report_csv'], index=False)
coverage_report = {
    'schema': 'geometry_labeling_coverage_report.v1',
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'visibility_mode': VISIBILITY_MODE,
    'frames': int(len(frame_report_df)),
    'status_counts': frame_report_df['status'].value_counts(dropna=False).to_dict() if not frame_report_df.empty else {},
    'projected_points_total': int(frame_report_df.get('projected_points', pd.Series(dtype=int)).sum()),
    'visible_points_total': int(frame_report_df.get('visible_points', pd.Series(dtype=int)).sum()),
    'rendered_depth_visible_total': int(frame_report_df.get('rendered_depth_visible_points', pd.Series(dtype=int)).sum()),
    'self_zbuffer_visible_total': int(frame_report_df.get('self_zbuffer_visible_points', pd.Series(dtype=int)).sum()),
    'mask_hits_total': int(frame_report_df.get('mask_hits', pd.Series(dtype=int)).sum()),
    'object_votes_total': int(frame_report_df.get('small_votes', pd.Series(dtype=int)).sum()),
    'small_votes_total': int(frame_report_df.get('small_votes', pd.Series(dtype=int)).sum()),  # Backward-compatible alias for object votes.
    'structural_votes_total': int(frame_report_df.get('structural_votes', pd.Series(dtype=int)).sum()),
    'structural_gate_rejections_total': int(frame_report_df.get('structural_gate_rejections', pd.Series(dtype=int)).sum()),
    'votes_total': int(frame_report_df.get('votes', pd.Series(dtype=int)).sum()),
    'foreground_rejected_masks_total': int(sum(value for key, value in role_competition_stats.items() if key.startswith('object_masks_rejected_foreground_'))),
}
role_competition_report = {
    'schema': 'geometry_labeling_role_competition_report.v1',
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'stats': {key: int(value) for key, value in role_competition_stats.items()},
    'config': {
        'enable_label_frequency_normalization': ENABLE_LABEL_FREQUENCY_NORMALIZATION,
        'label_frequency_power': LABEL_FREQUENCY_POWER,
        'label_frequency_min_weight': LABEL_FREQUENCY_MIN_WEIGHT,
        'label_frequency_max_weight': LABEL_FREQUENCY_MAX_WEIGHT,
        'median_label_mask_count': median_label_mask_count,
        'enable_mask_area_normalization': ENABLE_MASK_AREA_NORMALIZATION,
        'mask_area_reference_fraction': MASK_AREA_REFERENCE_FRACTION,
        'mask_area_power': MASK_AREA_POWER,
        'mask_area_min_weight': MASK_AREA_MIN_WEIGHT,
        'mask_area_max_weight': MASK_AREA_MAX_WEIGHT,
        'enable_gaussian_footprint_voting': ENABLE_GAUSSIAN_FOOTPRINT_VOTING,
        'gaussian_footprint_stddev_multiplier': GAUSSIAN_FOOTPRINT_STDDEV_MULTIPLIER,
        'gaussian_footprint_min_radius_px': GAUSSIAN_FOOTPRINT_MIN_RADIUS_PX,
        'gaussian_footprint_max_radius_px': GAUSSIAN_FOOTPRINT_MAX_RADIUS_PX,
        'gaussian_footprint_sample_pattern': GAUSSIAN_FOOTPRINT_SAMPLE_PATTERN,
        'gaussian_footprint_min_mask_fraction': GAUSSIAN_FOOTPRINT_MIN_MASK_FRACTION,
        'enable_object_foreground_geometry_check': ENABLE_OBJECT_FOREGROUND_GEOMETRY_CHECK,
        'object_foreground_min_delta_m': OBJECT_FOREGROUND_MIN_DELTA_M,
        'object_foreground_depth_percentile': OBJECT_FOREGROUND_DEPTH_PERCENTILE,
        'object_foreground_ring_pixels': OBJECT_FOREGROUND_RING_PIXELS,
        'object_foreground_min_valid_pixels': OBJECT_FOREGROUND_MIN_VALID_PIXELS,
        'object_foreground_exempt_labels': sorted(OBJECT_FOREGROUND_EXEMPT_LABELS),
        'small_object_vote_multiplier': SMALL_OBJECT_VOTE_MULTIPLIER,
        'structural_vote_multipliers': STRUCTURAL_VOTE_MULTIPLIERS,
        'default_structural_vote_multiplier': DEFAULT_STRUCTURAL_VOTE_MULTIPLIER,
        'small_mask_dilation_pixels': SMALL_MASK_DILATION_PIXELS,
        'structural_mask_dilation_pixels': STRUCTURAL_MASK_DILATION_PIXELS,
        'wall_mask_dilation_pixels': WALL_MASK_DILATION_PIXELS,
        'small_object_suppression_dilation_pixels': SMALL_OBJECT_SUPPRESSION_DILATION_PIXELS,
        'enable_structural_height_gates': ENABLE_STRUCTURAL_HEIGHT_GATES,
        'world_up_vector': WORLD_UP_VECTOR.astype(float).tolist(),
        'floor_height_limit': floor_height_limit,
        'ceiling_height_limit': ceiling_height_limit,
    }
}
OUTPUTS['coverage_report_json'].write_text(json.dumps(coverage_report, indent=2))
OUTPUTS['role_competition_report_json'].write_text(json.dumps(role_competition_report, indent=2))

print(frame_report_df['status'].value_counts(dropna=False))
print('Total raw point-mask votes:', int(frame_report_df.get('votes', pd.Series(dtype=int)).sum()))
print('Object votes:', coverage_report['object_votes_total'])
print('Structural votes:', coverage_report['structural_votes_total'])
print('Structural gate rejections:', coverage_report['structural_gate_rejections_total'])
print('Visibility mode:', VISIBILITY_MODE)
print('Coverage report:', OUTPUTS['coverage_report_json'])
print('Role competition report:', OUTPUTS['role_competition_report_json'])
print('Vote weight report:', OUTPUTS['vote_weight_report_csv'])
print('Mask geometry report:', OUTPUTS['mask_geometry_report_csv'])
print('Footprint vote report:', OUTPUTS['footprint_vote_report_csv'])
logger.info('Voting finished. Total raw point-mask votes: %d', int(frame_report_df.get('votes', pd.Series(dtype=int)).sum()))


## 10. Assign Final Point Labels

In [ ]:
# --- Cell explanation -----------------------------------------------------
# This cell converts raw per-label vote arrays into one final label per 3D point.
# It finds the best and second-best label score for every point, computes a confidence/margin,
# applies minimum vote/view/margin thresholds, and marks uncertain points as unknown (-1).
# It writes splat_semantic_labels.json, which is the authoritative per-point/per-splat metadata.
# It also writes geometry_labeling_label_summary.csv so label coverage can be inspected quickly.
# ------------------------------------------------------------------------

label_names = sorted(label_scores.keys())
label_to_id = {label: idx for idx, label in enumerate(label_names)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

best_label_id = np.full(len(points), -1, dtype=np.int32)
best_score = np.zeros(len(points), dtype=np.float32)
second_score = np.zeros(len(points), dtype=np.float32)

for label, scores in label_scores.items():
    label_id = label_to_id[label]
    better = scores > best_score
    second_score[better] = best_score[better]
    best_score[better] = scores[better]
    best_label_id[better] = label_id
    second_score[~better] = np.maximum(second_score[~better], scores[~better])

winning_views = np.zeros(len(points), dtype=np.uint16)
for label, views in label_views.items():
    label_id = label_to_id[label]
    winning_views[best_label_id == label_id] = views[best_label_id == label_id]

margin = np.zeros(len(points), dtype=np.float32)
valid_best = best_score > 0
margin[valid_best] = (best_score[valid_best] - second_score[valid_best]) / np.maximum(best_score[valid_best], 1e-6)
confidence = np.zeros(len(points), dtype=np.float32)
confidence[valid_best] = best_score[valid_best] / np.maximum(total_votes[valid_best], best_score[valid_best])


structural_cleanup_rejections = {}
if ENABLE_FINAL_STRUCTURAL_GEOMETRY_CLEANUP:
    for surface_label in SURFACE_STRUCTURAL_LABELS:
        label_id = label_to_id.get(surface_label)
        if label_id is None:
            continue
        candidate_idx = np.flatnonzero(best_label_id == label_id)
        if len(candidate_idx) == 0:
            structural_cleanup_rejections[surface_label] = 0
            continue
        gate = structural_geometry_gate(surface_label, candidate_idx)
        rejected_idx = candidate_idx[~gate]
        structural_cleanup_rejections[surface_label] = int(len(rejected_idx))
        if len(rejected_idx):
            best_label_id[rejected_idx] = -1
            best_score[rejected_idx] = 0.0
            second_score[rejected_idx] = 0.0
            margin[rejected_idx] = 0.0
            confidence[rejected_idx] = 0.0
            winning_views[rejected_idx] = 0

# A point becomes labeled only when the winning label is strong enough, supported by enough views,
# and clearly ahead of the second-best label. Otherwise it remains unknown.
labeled_mask = (
    (best_label_id >= 0)
    & (best_score >= MIN_TOTAL_VOTE)
    & (winning_views >= MIN_SUPPORT_VIEWS)
    & (margin >= MIN_LABEL_MARGIN)
)
final_label_id = np.where(labeled_mask, best_label_id, -1).astype(np.int32)

label_summary_rows = []
for label, label_id in label_to_id.items():
    idx = np.flatnonzero(final_label_id == label_id)
    label_summary_rows.append({
        'label': label,
        'point_count': int(len(idx)),
        'mean_confidence': float(confidence[idx].mean()) if len(idx) else 0.0,
        'mean_score': float(best_score[idx].mean()) if len(idx) else 0.0,
        'mean_support_views': float(winning_views[idx].mean()) if len(idx) else 0.0,
    })

label_summary_df = pd.DataFrame(label_summary_rows).sort_values('point_count', ascending=False)
label_summary_df.to_csv(OUTPUTS['label_summary_csv'], index=False)

point_label_package = {
    'schema': 'geometry_point_semantic_labels.v1',
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'target_ply': str(TARGET_PLY),
    'target_point_source': TARGET_POINT_SOURCE,
    'is_gaussian_splat': bool(is_gaussian_splat),
    'point_count': int(len(points)),
    'sampled_from_original': sample_indices.tolist() if sample_indices is not None else None,
    'labels': label_names,
    'unknown_label_id': -1,
    'label_ids': final_label_id.astype(int).tolist(),
    'confidence': np.round(confidence, 4).astype(float).tolist(),
    'support_views': winning_views.astype(int).tolist(),
    'labeled_splat_ply': str(OUTPUTS['labeled_splat_ply'].relative_to(SCAN_DIR)),
    'config': {
        'pipeline_version': PIPELINE_VERSION,
        'camera_convention': CAMERA_CONVENTION,
        'visibility_mode': VISIBILITY_MODE,
        'allow_no_depth_frame_votes': ALLOW_NO_DEPTH_FRAME_VOTES,
        'small_mask_dilation_pixels': SMALL_MASK_DILATION_PIXELS,
        'structural_mask_dilation_pixels': STRUCTURAL_MASK_DILATION_PIXELS,
        'wall_mask_dilation_pixels': WALL_MASK_DILATION_PIXELS,
        'small_object_suppression_dilation_pixels': SMALL_OBJECT_SUPPRESSION_DILATION_PIXELS,
        'enable_label_frequency_normalization': ENABLE_LABEL_FREQUENCY_NORMALIZATION,
        'label_frequency_power': LABEL_FREQUENCY_POWER,
        'label_frequency_min_weight': LABEL_FREQUENCY_MIN_WEIGHT,
        'label_frequency_max_weight': LABEL_FREQUENCY_MAX_WEIGHT,
        'median_label_mask_count': median_label_mask_count,
        'enable_mask_area_normalization': ENABLE_MASK_AREA_NORMALIZATION,
        'mask_area_reference_fraction': MASK_AREA_REFERENCE_FRACTION,
        'mask_area_power': MASK_AREA_POWER,
        'mask_area_min_weight': MASK_AREA_MIN_WEIGHT,
        'mask_area_max_weight': MASK_AREA_MAX_WEIGHT,
        'enable_gaussian_footprint_voting': ENABLE_GAUSSIAN_FOOTPRINT_VOTING,
        'gaussian_scale_is_log': GAUSSIAN_SCALE_IS_LOG,
        'gaussian_footprint_stddev_multiplier': GAUSSIAN_FOOTPRINT_STDDEV_MULTIPLIER,
        'gaussian_footprint_fallback_world_radius_m': GAUSSIAN_FOOTPRINT_FALLBACK_WORLD_RADIUS_M,
        'gaussian_footprint_min_radius_px': GAUSSIAN_FOOTPRINT_MIN_RADIUS_PX,
        'gaussian_footprint_max_radius_px': GAUSSIAN_FOOTPRINT_MAX_RADIUS_PX,
        'gaussian_footprint_sample_pattern': GAUSSIAN_FOOTPRINT_SAMPLE_PATTERN,
        'gaussian_footprint_min_mask_fraction': GAUSSIAN_FOOTPRINT_MIN_MASK_FRACTION,
        'has_gaussian_scale': bool(has_gaussian_scale),
        'enable_object_foreground_geometry_check': ENABLE_OBJECT_FOREGROUND_GEOMETRY_CHECK,
        'object_foreground_min_delta_m': OBJECT_FOREGROUND_MIN_DELTA_M,
        'object_foreground_depth_percentile': OBJECT_FOREGROUND_DEPTH_PERCENTILE,
        'object_foreground_ring_pixels': OBJECT_FOREGROUND_RING_PIXELS,
        'object_foreground_min_valid_pixels': OBJECT_FOREGROUND_MIN_VALID_PIXELS,
        'object_foreground_exempt_labels': sorted(OBJECT_FOREGROUND_EXEMPT_LABELS),
        'enable_final_structural_geometry_cleanup': ENABLE_FINAL_STRUCTURAL_GEOMETRY_CLEANUP,
        'structural_cleanup_rejections': structural_cleanup_rejections,
        'small_object_vote_multiplier': SMALL_OBJECT_VOTE_MULTIPLIER,
        'surface_structural_labels': sorted(SURFACE_STRUCTURAL_LABELS),
        'structural_vote_multipliers': STRUCTURAL_VOTE_MULTIPLIERS,
        'enable_structural_height_gates': ENABLE_STRUCTURAL_HEIGHT_GATES,
        'world_up_vector': WORLD_UP_VECTOR.astype(float).tolist(),
        'floor_height_limit': floor_height_limit,
        'ceiling_height_limit': ceiling_height_limit,
        'depth_patch_radius_px': DEPTH_PATCH_RADIUS_PX,
        'rendered_depth_vote_weight': RENDERED_DEPTH_VOTE_WEIGHT,
        'self_zbuffer_vote_weight': SELF_ZBUFFER_VOTE_WEIGHT,
        'no_depth_vote_weight': NO_DEPTH_VOTE_WEIGHT,
        'depth_abs_tolerance_m': DEPTH_ABS_TOLERANCE_M,
        'depth_rel_tolerance': DEPTH_REL_TOLERANCE,
        'self_zbuffer_abs_tolerance_m': SELF_ZBUFFER_ABS_TOLERANCE_M,
        'self_zbuffer_rel_tolerance': SELF_ZBUFFER_REL_TOLERANCE,
        'self_zbuffer_radius_px': SELF_ZBUFFER_RADIUS_PX,
        'min_total_vote': MIN_TOTAL_VOTE,
        'min_support_views': MIN_SUPPORT_VIEWS,
        'min_label_margin': MIN_LABEL_MARGIN,
    }
}
OUTPUTS['labels_json'].write_text(json.dumps(point_label_package))
augmented_splat_written = write_augmented_gaussian_ply(
    TARGET_PLY,
    OUTPUTS['labeled_splat_ply'],
    final_label_id,
    confidence,
    winning_views,
)

print('Labeled points:', int(labeled_mask.sum()), '/', len(points))
print('Augmented labeled splat PLY written:', augmented_splat_written, OUTPUTS['labeled_splat_ply'])
print('Unknown/unlabeled points:', int((~labeled_mask).sum()))
print('Final structural cleanup rejections:', structural_cleanup_rejections)
display(label_summary_df.head(30))
logger.info('Assigned labels to %d / %d points', int(labeled_mask.sum()), len(points))


## 11. Cluster Labeled Points Into Viewer Objects

In [ ]:
# --- Cell explanation -----------------------------------------------------
# This cell groups labeled points into viewer-friendly object proxies.
# Structural surfaces are kept as one object per label, while object-like labels are clustered with DBSCAN
# so multiple chairs/desks/etc. can become separate proxy objects when geometry supports it.
# For each object, the cell computes a robust bounding box, writes an individual PLY, and contributes
# to the combined semantic-colored point cloud and bounding-box PLY used by the local viewer.
# It exports semantic_scene_manifest.json/csv and object_bounding_boxes.json/ply.
# ------------------------------------------------------------------------

rng = np.random.default_rng(RANDOM_SEED)
small_dir = POINTS_DIR / 'small'
structural_dir = POINTS_DIR / 'structural'
small_dir.mkdir(parents=True, exist_ok=True)
structural_dir.mkdir(parents=True, exist_ok=True)

objects = []
combined_points = []
combined_colors = []
bbox_points = []
bbox_colors = []

label_roles = mask_df.groupby('label')['role'].agg(lambda s: 'structural' if (s == 'structural').any() else 'small').to_dict()


# Convert per-point semantic labels into object-level proxy groups for the viewer.
for label in tqdm(label_names, desc='Clustering labels into objects'):
    label_id = label_to_id[label]
    label_idx = np.flatnonzero(final_label_id == label_id)
    if len(label_idx) < MIN_OBJECT_POINTS:
        continue

    role = label_roles.get(label, 'structural' if label in STRUCTURAL_LABELS else 'small')
    object_groups = []

    if role == 'structural' or label in STRUCTURAL_LABELS:
        object_groups = [(0, label_idx)]
    else:
        clustering_idx = label_idx
        if len(clustering_idx) > MAX_CLUSTER_POINTS_PER_LABEL:
            clustering_idx = np.sort(rng.choice(clustering_idx, size=MAX_CLUSTER_POINTS_PER_LABEL, replace=False))
        coords = points[clustering_idx]
        cluster_labels = DBSCAN(eps=DBSCAN_EPS_M, min_samples=DBSCAN_MIN_SAMPLES).fit_predict(coords)
        for cluster_id in sorted(set(cluster_labels)):
            if cluster_id == -1:
                continue
            member_idx = clustering_idx[cluster_labels == cluster_id]
            if len(member_idx) >= MIN_OBJECT_POINTS:
                object_groups.append((int(cluster_id), member_idx))
        if not object_groups:
            object_groups = [(0, label_idx)]

    color = stable_label_color(label)
    for object_number, (_, idx) in enumerate(object_groups, start=1):
        object_points = points[idx]
        if len(object_points) < MIN_OBJECT_POINTS:
            continue
        robust_min = np.percentile(object_points, 2, axis=0)
        robust_max = np.percentile(object_points, 98, axis=0)
        raw_min = object_points.min(axis=0)
        raw_max = object_points.max(axis=0)
        center = ((robust_min + robust_max) / 2.0).astype(float)
        size = np.maximum(robust_max - robust_min, 1e-4).astype(float)
        object_id = f'{role}_{label}_{object_number:03d}'
        object_dir = structural_dir if role == 'structural' else small_dir
        ply_path = object_dir / f'{object_id}.ply'
        object_rgb = np.tile(color.reshape(1, 3), (len(object_points), 1))
        write_point_ply(ply_path, object_points, object_rgb)

        objects.append({
            'object_id': object_id,
            'label': label,
            'role': role,
            'point_count': int(len(idx)),
            'confidence': float(confidence[idx].mean()),
            'support_views': float(winning_views[idx].mean()),
            'max_support_views': int(winning_views[idx].max()) if len(idx) else 0,
            'center': center.tolist(),
            'size': size.tolist(),
            'min': robust_min.astype(float).tolist(),
            'max': robust_max.astype(float).tolist(),
            'raw_min': raw_min.astype(float).tolist(),
            'raw_max': raw_max.astype(float).tolist(),
            'color': color.astype(int).tolist(),
            'ply_path': str(ply_path.relative_to(SCAN_DIR)),
        })

        combined_points.append(object_points)
        combined_colors.append(object_rgb)
        edges = box_edges_from_min_max(robust_min, robust_max)
        bbox_points.append(edges)
        bbox_colors.append(np.tile(color.reshape(1, 3), (len(edges), 1)))

if not objects:
    fail('No viewer objects were produced. Lower thresholds or inspect geometry_labeling_label_summary.csv.')

semantic_df = pd.DataFrame(objects).sort_values(['role', 'label', 'point_count'], ascending=[True, True, False])
semantic_df.to_csv(OUTPUTS['semantic_manifest_csv'], index=False)

manifest = {
    'schema': 'semantic_scene_manifest.v2.geometry_labeling',
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'scan_id': SCAN_ID,
    'target_ply': str(TARGET_PLY),
    'point_label_file': str(OUTPUTS['labels_json'].relative_to(SCAN_DIR)),
    'objects': objects,
}
OUTPUTS['semantic_manifest_json'].write_text(json.dumps(manifest, indent=2))

bbox_payload = {
    'schema': 'object_bounding_boxes.v2.geometry_labeling',
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'scan_id': SCAN_ID,
    'boxes': [
        {
            'object_id': obj['object_id'],
            'label': obj['label'],
            'role': obj['role'],
            'center': obj['center'],
            'size': obj['size'],
            'min': obj['min'],
            'max': obj['max'],
            'confidence': obj['confidence'],
            'support_views': obj['support_views'],
            'point_count': obj['point_count'],
            'color': obj['color'],
        }
        for obj in objects
    ],
}
OUTPUTS['bbox_json'].write_text(json.dumps(bbox_payload, indent=2))

# Export combined semantic point preview. Sample if necessary to keep browser inspection practical.
all_combined_points = np.concatenate(combined_points, axis=0)
all_combined_colors = np.concatenate(combined_colors, axis=0)
if len(all_combined_points) > MAX_PREVIEW_POINTS:
    preview_idx = np.sort(rng.choice(len(all_combined_points), size=MAX_PREVIEW_POINTS, replace=False))
    all_combined_points = all_combined_points[preview_idx]
    all_combined_colors = all_combined_colors[preview_idx]
write_point_ply(OUTPUTS['semantic_ply'], all_combined_points, all_combined_colors)

all_bbox_points = np.concatenate(bbox_points, axis=0)
all_bbox_colors = np.concatenate(bbox_colors, axis=0)
write_point_ply(OUTPUTS['bbox_ply'], all_bbox_points, all_bbox_colors)

display(semantic_df.head(40))
print('Objects:', len(objects))
print('Semantic manifest JSON:', OUTPUTS['semantic_manifest_json'])
print('Object boxes JSON:', OUTPUTS['bbox_json'])
print('Semantic-colored PLY:', OUTPUTS['semantic_ply'])
print('Object boxes PLY:', OUTPUTS['bbox_ply'])
logger.info('Exported %d objects', len(objects))


## 12. Summary and Viewer Handoff

In [ ]:
# --- Cell explanation -----------------------------------------------------
# This final cell writes a compact run summary and prints the local viewer handoff command.
# The summary records the scan id, target PLY, number of labeled/unlabeled points, object count,
# processed/skipped frame counts, output paths, and the active thresholds/configuration.
# Use geometry_labeling_summary.json with the frame, coverage, role-competition, and label reports
# to compare projection runs and diagnose whether labels are under- or over-covering the room.
# ------------------------------------------------------------------------

summary = {
    'schema': 'geometry_labeling_summary.v1',
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'scan_id': SCAN_ID,
    'target_ply': str(TARGET_PLY),
    'target_point_source': TARGET_POINT_SOURCE,
    'is_gaussian_splat': bool(is_gaussian_splat),
    'point_count': int(len(points)),
    'labeled_point_count': int(labeled_mask.sum()),
    'unlabeled_point_count': int((~labeled_mask).sum()),
    'label_count': int(len(label_names)),
    'object_count': int(len(objects)),
    'frames_total': int(len(frame_groups)),
    'frames_processed': int((frame_report_df['status'] == 'processed').sum()) if not frame_report_df.empty else 0,
    'frames_skipped_no_depth': int((frame_report_df['status'] == 'skipped_no_depth').sum()) if not frame_report_df.empty else 0,
    'outputs': {key: str(value) for key, value in OUTPUTS.items()},
    'augmented_splat_written': bool(augmented_splat_written),
    'config': {
        'pipeline_version': PIPELINE_VERSION,
        'camera_convention': CAMERA_CONVENTION,
        'visibility_mode': VISIBILITY_MODE,
        'allow_no_depth_frame_votes': ALLOW_NO_DEPTH_FRAME_VOTES,
        'small_mask_dilation_pixels': SMALL_MASK_DILATION_PIXELS,
        'structural_mask_dilation_pixels': STRUCTURAL_MASK_DILATION_PIXELS,
        'wall_mask_dilation_pixels': WALL_MASK_DILATION_PIXELS,
        'small_object_suppression_dilation_pixels': SMALL_OBJECT_SUPPRESSION_DILATION_PIXELS,
        'enable_label_frequency_normalization': ENABLE_LABEL_FREQUENCY_NORMALIZATION,
        'label_frequency_power': LABEL_FREQUENCY_POWER,
        'label_frequency_min_weight': LABEL_FREQUENCY_MIN_WEIGHT,
        'label_frequency_max_weight': LABEL_FREQUENCY_MAX_WEIGHT,
        'median_label_mask_count': median_label_mask_count,
        'enable_mask_area_normalization': ENABLE_MASK_AREA_NORMALIZATION,
        'mask_area_reference_fraction': MASK_AREA_REFERENCE_FRACTION,
        'mask_area_power': MASK_AREA_POWER,
        'mask_area_min_weight': MASK_AREA_MIN_WEIGHT,
        'mask_area_max_weight': MASK_AREA_MAX_WEIGHT,
        'enable_gaussian_footprint_voting': ENABLE_GAUSSIAN_FOOTPRINT_VOTING,
        'gaussian_scale_is_log': GAUSSIAN_SCALE_IS_LOG,
        'gaussian_footprint_stddev_multiplier': GAUSSIAN_FOOTPRINT_STDDEV_MULTIPLIER,
        'gaussian_footprint_fallback_world_radius_m': GAUSSIAN_FOOTPRINT_FALLBACK_WORLD_RADIUS_M,
        'gaussian_footprint_min_radius_px': GAUSSIAN_FOOTPRINT_MIN_RADIUS_PX,
        'gaussian_footprint_max_radius_px': GAUSSIAN_FOOTPRINT_MAX_RADIUS_PX,
        'gaussian_footprint_sample_pattern': GAUSSIAN_FOOTPRINT_SAMPLE_PATTERN,
        'gaussian_footprint_min_mask_fraction': GAUSSIAN_FOOTPRINT_MIN_MASK_FRACTION,
        'has_gaussian_scale': bool(has_gaussian_scale),
        'enable_object_foreground_geometry_check': ENABLE_OBJECT_FOREGROUND_GEOMETRY_CHECK,
        'object_foreground_min_delta_m': OBJECT_FOREGROUND_MIN_DELTA_M,
        'object_foreground_depth_percentile': OBJECT_FOREGROUND_DEPTH_PERCENTILE,
        'object_foreground_ring_pixels': OBJECT_FOREGROUND_RING_PIXELS,
        'object_foreground_min_valid_pixels': OBJECT_FOREGROUND_MIN_VALID_PIXELS,
        'object_foreground_exempt_labels': sorted(OBJECT_FOREGROUND_EXEMPT_LABELS),
        'enable_final_structural_geometry_cleanup': ENABLE_FINAL_STRUCTURAL_GEOMETRY_CLEANUP,
        'structural_cleanup_rejections': structural_cleanup_rejections,
        'small_object_vote_multiplier': SMALL_OBJECT_VOTE_MULTIPLIER,
        'surface_structural_labels': sorted(SURFACE_STRUCTURAL_LABELS),
        'structural_vote_multipliers': STRUCTURAL_VOTE_MULTIPLIERS,
        'enable_structural_height_gates': ENABLE_STRUCTURAL_HEIGHT_GATES,
        'world_up_vector': WORLD_UP_VECTOR.astype(float).tolist(),
        'floor_height_limit': floor_height_limit,
        'ceiling_height_limit': ceiling_height_limit,
        'depth_patch_radius_px': DEPTH_PATCH_RADIUS_PX,
        'rendered_depth_vote_weight': RENDERED_DEPTH_VOTE_WEIGHT,
        'self_zbuffer_vote_weight': SELF_ZBUFFER_VOTE_WEIGHT,
        'no_depth_vote_weight': NO_DEPTH_VOTE_WEIGHT,
        'depth_abs_tolerance_m': DEPTH_ABS_TOLERANCE_M,
        'depth_rel_tolerance': DEPTH_REL_TOLERANCE,
        'self_zbuffer_abs_tolerance_m': SELF_ZBUFFER_ABS_TOLERANCE_M,
        'self_zbuffer_rel_tolerance': SELF_ZBUFFER_REL_TOLERANCE,
        'self_zbuffer_radius_px': SELF_ZBUFFER_RADIUS_PX,
        'min_total_vote': MIN_TOTAL_VOTE,
        'min_support_views': MIN_SUPPORT_VIEWS,
        'min_label_margin': MIN_LABEL_MARGIN,
        'dbscan_eps_m': DBSCAN_EPS_M,
        'dbscan_min_samples': DBSCAN_MIN_SAMPLES,
    },
}
OUTPUTS['summary_json'].write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

print('\nOpen locally after syncing/downloading outputs:')
print('node viewer/server.js --data-root "/path/to/room-model-project" --scan scan_001')
print('http://localhost:5177/viewer/')
